# Time Series Econometrics — ARIMA & GARCH on S&P 500 (SPY)

**Goal:** Explore what can realistically be modeled in a financial time series.
Rather than predicting market direction, this notebook focuses on understanding the **temporal structure** of returns and modeling **conditional volatility**.

**Key question:** Can we model risk even when returns are unpredictable?

**Dataset:** SPY daily closing prices from Yahoo Finance (2015–2024)

## 1. Setup & Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error
from arch import arch_model

plt.rcParams["figure.figsize"] = (12, 5)

In [ ]:
# Download SPY daily data (2015–2024)
data = yf.download("SPY", start="2015-01-01", end="2024-01-01", auto_adjust=False)
data = data[["Close", "Volume"]].dropna()

print(f"Dataset: {data.shape[0]} trading days")
data.head()

## 2. Prices vs. Returns

Financial prices are typically **non-stationary**: they trend upward over time with growing variance.
Returns (percentage changes) remove this trend and produce a more stationary series — a prerequisite for ARIMA modeling.

$$r_t = \frac{P_t - P_{t-1}}{P_{t-1}}$$

In [ ]:
price = data["Close"]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(price)
axes[0].set_title("SPY Closing Price — Non-stationary (upward trend)")
axes[0].set_ylabel("Price ($)")
axes[0].grid(True)

returns = price.pct_change().dropna()
axes[1].plot(returns)
axes[1].set_title("SPY Daily Returns — More stationary (mean-reverting)")
axes[1].set_ylabel("Return")
axes[1].grid(True)

plt.tight_layout()
plt.show()

print(f"Returns — Mean: {returns.mean().values[0]:.4f} | Std: {returns.std().values[0]:.4f}")

In [ ]:
# Return distribution — check for fat tails and asymmetry
returns.hist(bins=60)
plt.title("Daily Return Distribution — Fat tails expected (leptokurtosis)")
plt.xlabel("Return")
plt.ylabel("Frequency")
plt.grid(True)
plt.show()

pos = (returns > 0).sum().values[0]
neg = (returns < 0).sum().values[0]
print(f"Positive days: {pos} ({pos/(pos+neg):.1%}) | Negative days: {neg} ({neg/(pos+neg):.1%})")

## 3. Stationarity Testing — Augmented Dickey-Fuller (ADF)

The ADF test checks whether a series has a unit root (non-stationary, H0) or not.
- **p > 0.05**: fail to reject H0 → series is non-stationary
- **p < 0.05**: reject H0 → series is stationary

In [ ]:
def adf_test(series, name):
    """Run ADF test and print a clean summary."""
    result = adfuller(series.dropna())
    p = result[1]
    conclusion = "STATIONARY" if p < 0.05 else "NON-STATIONARY"
    print(f"{name}: ADF stat = {result[0]:.4f} | p-value = {p:.2e} → {conclusion}")

adf_test(price.squeeze(),   "Prices  ")
adf_test(returns.squeeze(), "Returns ")

## 4. Volatility Analysis — Clustering & Risk Regimes

Even if returns are stationary in mean, their **variance is not constant over time**.
Periods of high volatility tend to cluster together — a phenomenon known as **volatility clustering**.
This is the core motivation for GARCH modeling.

In [ ]:
rolling_vol = returns.rolling(window=20).std()
max_vol_date = rolling_vol.idxmax()

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

axes[0].plot(returns.abs())
axes[0].set_title("|Returns| — Volatility clustering visible")
axes[0].set_ylabel("|r_t|")
axes[0].grid(True)

axes[1].plot(rolling_vol, label="20-day rolling volatility")
axes[1].axvline(max_vol_date, color='red', linestyle='--',
                label=f"Peak: {max_vol_date[0].strftime('%b %Y')} (COVID crash)")
axes[1].set_title("Rolling Volatility — Heteroscedastic (non-constant variance)")
axes[1].set_ylabel("Volatility")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

print(f"Peak volatility: {rolling_vol.max().values[0]:.4f} on {max_vol_date[0].strftime('%Y-%m-%d')}")

## 5. Autocorrelation Analysis — Returns vs. Squared Returns

A key empirical finding in financial markets:
- **ACF of r_t**: near-zero at all lags → returns are close to white noise → hard to predict
- **ACF of r_t²**: significant and persistent → volatility has memory → partially predictable

This asymmetry is the statistical foundation for ARIMA (conditional mean) vs. GARCH (conditional variance).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

plot_acf(returns.squeeze(), lags=30, ax=axes[0])
axes[0].set_title("ACF of Returns r_t\nNear-zero → little temporal structure")

plot_acf(returns.squeeze()**2, lags=30, ax=axes[1])
axes[1].set_title("ACF of Squared Returns r_t²\nPersistent → volatility has memory")

plt.tight_layout()
plt.show()

acf_lag1 = returns.squeeze().autocorr(lag=1)
acf2_lag1 = (returns.squeeze()**2).autocorr(lag=1)
print(f"Autocorrelation at lag 1 — Returns: {acf_lag1:.4f} | Squared returns: {acf2_lag1:.4f}")

## 6. ARIMA Modeling — Conditional Mean

ARIMA(p, d, q) models the **conditional mean** of the series using past values and past errors.
Given the near-zero autocorrelation of returns, ARIMA is expected to find very little signal.

In [ ]:
# Chronological train/test split — no shuffle
split = int(len(returns) * 0.8)
train_returns = returns.iloc[:split]
test_returns  = returns.iloc[split:]

print(f"Train: {len(train_returns)} days ({train_returns.index[0].date()} → {train_returns.index[-1].date()})")
print(f"Test:  {len(test_returns)} days ({test_returns.index[0].date()} → {test_returns.index[-1].date()})")

In [ ]:
# Fit ARIMA(1,0,1) on training returns
arima_result = ARIMA(train_returns, order=(1, 0, 1)).fit()
print(arima_result.summary())

In [ ]:
# Compare ARIMA configurations by AIC and test MSE
configs = [(1,0,1), (2,0,1), (1,0,2), (2,0,2), (3,0,1)]
arima_results = []

for order in configs:
    try:
        m = ARIMA(train_returns, order=order).fit()
        fc = m.forecast(steps=len(test_returns))
        mse = mean_squared_error(test_returns, fc)
        arima_results.append({"Order": str(order), "AIC": round(m.aic, 2), "Test MSE": round(mse, 8)})
    except Exception:
        pass

pd.DataFrame(arima_results).sort_values("AIC")

In [ ]:
# Forecast vs. actual returns on the test set
forecast = arima_result.forecast(steps=len(test_returns))

plt.figure(figsize=(12, 4))
plt.plot(test_returns.index, test_returns.values, label="Actual returns", alpha=0.7)
plt.plot(test_returns.index, forecast.values, label="ARIMA forecast", alpha=0.9)
plt.title("ARIMA Forecast vs. Actual Returns\nForecasts converge to ~0 — very little signal in returns")
plt.xlabel("Date")
plt.ylabel("Return")
plt.legend()
plt.grid(True)
plt.show()

## 7. Residual Diagnostics — ARCH Effects

After fitting ARIMA, we examine the residuals to check whether any temporal structure remains unexploited.
- If ACF(residuals) is flat → linear structure fully captured ✓
- If ACF(residuals²) shows persistence → **ARCH effects present** → GARCH needed

In [ ]:
residuals = arima_result.resid

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(residuals)
axes[0].set_title("ARIMA Residuals over time")
axes[0].set_ylabel("Residual")
axes[0].grid(True)

plot_acf(residuals, lags=30, ax=axes[1])
axes[1].set_title("ACF of Residuals\nFlat → linear structure captured")

plot_acf(residuals**2, lags=30, ax=axes[2])
axes[2].set_title("ACF of Squared Residuals\nPersistent → ARCH effects remain")

plt.tight_layout()
plt.show()

## 8. GARCH(1,1) — Conditional Variance Modeling

GARCH models the **time-varying conditional variance** $h_t$:

$$h_t = \omega + \alpha \cdot \varepsilon_{t-1}^2 + \beta \cdot h_{t-1}$$

- $\omega$ : baseline variance (long-run level)
- $\alpha$ : reactivity to recent shocks
- $\beta$ : persistence of past variance
- $\alpha + \beta$ close to 1 → very high volatility persistence

In [ ]:
# GARCH works better with returns expressed in percent
returns_pct = 100 * returns

garch_model  = arch_model(returns_pct, vol="Garch", p=1, q=1)
garch_result = garch_model.fit(disp="off")
print(garch_result.summary())

In [ ]:
# Extract and interpret GARCH parameters
params = garch_result.params
alpha = params.get('alpha[1]', params.iloc[2])
beta  = params.get('beta[1]',  params.iloc[3])

print(f"alpha (shock reactivity):    {alpha:.4f}")
print(f"beta  (variance persistence): {beta:.4f}")
print(f"alpha + beta (total persistence): {alpha + beta:.4f}")
print(f"→ Volatility shocks take ~{1/(1-(alpha+beta)):.0f}x longer to dissipate than a white noise process")

In [ ]:
# Conditional volatility estimated by GARCH
cond_vol = garch_result.conditional_volatility
max_vol_date = cond_vol.idxmax()

plt.figure(figsize=(12, 5))
plt.plot(cond_vol, label="GARCH(1,1) conditional volatility", linewidth=1)
plt.axvline(max_vol_date, color='red', linestyle='--',
            label=f"Peak: {max_vol_date.strftime('%b %Y')} — COVID crash")
plt.title("Conditional Volatility Estimated by GARCH(1,1)")
plt.xlabel("Date")
plt.ylabel("Conditional Volatility (%)")
plt.legend()
plt.grid(True)
plt.show()

print(f"Peak conditional volatility: {cond_vol.max():.2f}% on {max_vol_date.strftime('%Y-%m-%d')}")

## 9. ARIMA vs. GARCH — Summary Comparison

In [ ]:
comparison = pd.DataFrame([
    {
        "Model":    "ARIMA",
        "Models":   "Conditional mean E[r_t | F_{t-1}]",
        "Signal":   "Weak (ACF of r_t ≈ 0)",
        "Use case": "Return forecasting",
        "Limitation": "Linear, constant variance assumption"
    },
    {
        "Model":    "GARCH(1,1)",
        "Models":   "Conditional variance h_t = Var(ε_t | F_{t-1})",
        "Signal":   "Strong (ACF of r_t² persistent)",
        "Use case": "Volatility & risk forecasting",
        "Limitation": "Gaussian tails, no structural breaks"
    }
])
comparison.set_index("Model")

## 10. Conclusion

**Key findings:**

- **Prices are non-stationary**, returns are stationary (confirmed by ADF test: p ≈ 1e-27)
- **Returns have near-zero autocorrelation** → ARIMA finds almost no exploitable signal → forecasts converge to ~0
- **Squared returns are strongly autocorrelated** → volatility has persistent memory
- **GARCH(1,1)** successfully captures this volatility clustering: α + β ≈ 0.975 (very high persistence)
- **Peak volatility: March 2020** (COVID crash) — detected automatically from the GARCH conditional variance

**Core insight:**
> Predicting the *direction* of financial returns is near-impossible (random walk).  
> Predicting *volatility* is substantially more realistic — it has real temporal memory.  
> This is why risk modeling (VaR, option pricing, portfolio management) relies on GARCH-type models rather than return forecasters.